# Solar Flux Prediction

In [ ]:
import torch

device_try = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device_try}")

In [ ]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux (it runs in the global terminal).")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
print("--- Caricamento e Aggregazione Mensile ---")
# Carichiamo il dataset originale (non quello con il lag precedente)
df = pd.read_csv("Data/MARSIS_historical_dataset.csv", sep=";")
df.columns = df.columns.str.strip()

# Convertiamo l'Ephemeris Time (secondi dal J2000) in oggetti Datetime reali
df['datetime'] = pd.to_datetime(df['FM_data_ephemeris_time'], unit='s', origin=pd.Timestamp('2000-01-01 12:00:00'))

# Aggreghiamo su base mensile calcolando la media del flusso solare
df_monthly = df.groupby(df['datetime'].dt.to_period('M')).agg({
    'FM_data_F10_7_index': 'mean'
}).reset_index()

df_monthly['datetime'] = df_monthly['datetime'].dt.to_timestamp()
# Estraiamo il mese come feature numerica (timestamp relativo)
df_monthly['month_of_year'] = df_monthly['datetime'].dt.month

print(f"Dataset mensile creato. Totale mesi disponibili: {len(df_monthly)}")